In [302]:
from astropy.io import fits
import spiceypy as spice
import numpy as np

In [303]:
class Quaternion:
    def __init__(self, array=None, phi=0.0, axis=0):
        if array is None:
            array = np.array([0, 0, 0, np.cos(phi / 2)])
            array[axis] = np.sin(phi / 2)
            self.array = array
        else:
            self.array = np.array(array)

        self.v = self.array[0:3]
        self.s = self.array[3]
        self.x = self.v[0]
        self.y = self.v[1]
        self.z = self.v[2]

        self.spice = np.array([self.s, -self.v[0], -self.v[1], -self.v[2]])

    def __add__(self, other):
        return Quaternion(self.array + other.array)

    def __mul__(self, other):
        v = self.s * other.v + other.s * self.v - np.cross(self.v, other.v)
        s = self.s * other.s - np.dot(self.v, other.v)
        return Quaternion([v[0], v[1], v[2], s])

In [304]:
# Transform JD to Gregorian date
def jd2ymd(jd):
    l = int(jd + 68569)
    n = (4 * l) // 146097
    l = l - ((146097 * n + 3) // 4)
    i = (4000 * (l + 1)) // 1461001
    l = l - (1461 * i) // 4 + 31
    j = (80 * l) // 2447
    d = l - (2447 * j) // 80
    l = j // 11
    m = j + 2 - (12 * l)
    y = (100 * (n - 49)) + i + l
    return y, m, d


# Convert CODEC PCS timastamp to UTC
def timeFromPcsTimeStamp(timestamp=(0, 0)):
    seconds, subseconds = timestamp
    epochJd = 2440588  # Julian Day of 1970-01-01
    epochLeap = (
        27  # leap seconds since epoch (PCS sticks this in, so we need to remove it)
    )

    # Convert 16-bit subseconds to integer microseconds
    microseconds = int(15.2587890625 * subseconds)

    # Calculate Integer Julian Day from PCS seconds since epoch
    julianDay = epochJd + int(seconds - epochLeap) // 86400

    # Convert Julian Day to Gregorian Date
    year, month, day = jd2ymd(julianDay)

    # Calculate time of day in seconds from seconds since epoch using
    timeOfDay = int(seconds - epochLeap) % 86400

    # Convert time of day to hours, minutes, seconds
    remainder = int(timeOfDay)
    hour = remainder // 3600
    remainder = remainder % 3600
    minute = remainder // 60
    second = remainder % 60

    # Create UTC ISO string
    utc = "{0:04d}-{1:02d}-{2:02d}T{3:02d}:{4:02d}:{5:06.3f}Z".format(
        year, month, day, hour, minute, second + microseconds / 1e6
    )

    return utc


# Transform celestial coordinates to CODEX projective coordinates
def radec2hpc(et, r, ra=0, dec=0, **kwargs):
    object = kwargs.get("object", None)
    n = np.size(ra)
    c = np.empty([3, n])

    if object is None:
        # Compute rect. coordinates from RA and Dec
        c[0] = np.cos(dec) * np.cos(ra)
        c[1] = np.cos(dec) * np.sin(ra)
        c[2] = np.sin(dec)
    else:
        # Compute rect. coordinates of the desired object in J2000 frame
        s, lt = spice.spkezr(object, et, "j2000", "none", "earth")
        c[:, 0] = s[0:3]

    for i in range(n):
        # Rotate coordinates from J2000 to CODEX frame
        c[:, i] = np.array(spice.reclat(r.dot(c[:, i])))

    # Change the sign of the longitude (TBC why it is needed)
    c[1] = -c[1]

    return c[1:3]


# Compute rotation matrix for a desired angle
def rot(x=0):
    r = np.array([[np.cos(x), -np.sin(x)], [np.sin(x), np.cos(x)]])

    return r

In [305]:
# Inputs - they may be read from a json file

# L0 image file and SPICE metakernel
input = {
    "l0_image": "data/codex_l0_20250103_144920_4_2.fits",
    "spice_kernel": "metakernel.tm",
}

# Telemetered quaternions and gimbal angles
boresite = {
    "qPI": [0.86901498, -0.27779409, -0.34568208, -0.21723367],  # from msg ID 9
    "qBHr": [2.94100939e-08, -3.87863111e-07, 6.20560695e-06, 1.0],  # from msg ID 5
    "Elg": -21.08327866,  # elevetion in degrees
    "Azg": 39.22232866,  # azimuth in degrees
}

# Multiple ground measurements taken to characterize the misalignment parameters
angles = {"Azer": -1.708, "epsx": -0.087, "alfx": 0.013, "alfy": -0.306}

# WCS parameters derived from star fiels of 2025-01-03 14:49:20 image
wcs = {
    "crpix": [1971.722015853095, 1075.167887220806],  # boresgiht pixel
    "cdelt": 0.001810039520077789,  # plate scale in degrees
    "crota": -1.6962362045362835,  # detector tilt in degrees
}

In [306]:
# # Inputs - they may be read from a json file

# # L0 image file and SPICE metakernel
# input = {
#     "l0_image": "data/codex_l0_20250103_195222_4_2.fits",
#     "spice_kernel": "metakernel.tm",
# }

# # Telemetered quaternions and gimbal angles
# boresite = {
#     "qPI": [0.7869615, 0.41924217, -0.06860562, -0.44629639],  # from msg ID 9
#     "qBHr": [3.10666906e-08, -1.25082061e-05, -1.31869438e-05, 1.0],  # from msg ID 5
#     "Elg": -27.58892298,  # elevetion in degrees
#     "Azg": 131.42364979,  # azimuth in degrees
# }

# # Multiple ground measurements taken to characterize the misalignment parameters
# angles = {"Azer": -1.708, "epsx": -0.087, "alfx": 0.013, "alfy": -0.306}

# # WCS parameters derived from star fiels of 2025-01-03 14:49:20 image
# wcs = {
#     "crpix": [1971.722015853095, 1075.167887220806],  # boresgiht pixel
#     "cdelt": 0.001810039520077789,  # plate scale in degrees
#     "crota": -1.6962362045362835,  # detector tilt in degrees
# }

In [307]:
spice.furnsh(input["spice_kernel"])

In [308]:
qPI = Quaternion(boresite["qPI"])
qBHr = Quaternion(boresite["qBHr"])

Elg = np.radians(boresite["Elg"])
Azg = np.radians(boresite["Azg"])
Azer = np.radians(angles["Azer"])
epsx = np.radians(angles["epsx"])
alfx = np.radians(angles["alfx"])
alfy = np.radians(angles["alfy"])

crpix = wcs["crpix"]
cdelt = wcs["cdelt"]
crota = np.radians(wcs["crota"])

In [309]:
# # Testing engineering quaternions algebra

# # - define three engineering quaternions
# phi = np.radians(60)
# p = Quaternion(phi=phi, axis=0)
# q = Quaternion(phi=phi / 2, axis=1)
# s = Quaternion(phi=phi / 4, axis=2)

# # - make product
# m = p * q * s

# # - converto to spice quaternion and print rotation matrix
# print(spice.q2m(m.spice))

# # - print product of rotation matrices from spice quaternions
# print(spice.q2m(p.spice).dot((spice.q2m(q.spice).dot(spice.q2m(s.spice)))))

In [310]:
hdu = fits.open(input["l0_image"])
image = np.float64(hdu[0].data)
header = hdu[0].header

# Vertically flip image to restore CODEX viewpoint
image = np.flip(image, axis=1)

In [311]:
qBP = (
    Quaternion(phi=-Elg, axis=1)
    * Quaternion(phi=epsx, axis=0)
    * Quaternion(phi=Azg, axis=2)
    * Quaternion(phi=alfx, axis=0)
    * Quaternion(phi=alfy, axis=1)
    * Quaternion(phi=Azer, axis=2)
)

qBI = qBP * qPI

# From a solar ephemeris, we can compute a zero-bank quaternion that points the x-axis at the Sun

et = spice.str2et(header["obt_beg"])

# Get the position of the Sun as seen by Earth in J2000 at ET
s, light_time = spice.spkpos("sun", et, "j2000", "none", "earth")

# Get RA and Dec of the Sun
sunr, sunra, sundec = spice.recrad(s)

qH0I = Quaternion(phi=-sundec, axis=1) * Quaternion(phi=sunra, axis=2)

In [312]:
# Calculate coefficients a and b, and angle phi as described in Jim's procedure

a = (
    qBHr.s * qBI.s * qH0I.s
    + qBHr.s * qBI.x * qH0I.x
    + qBHr.s * qBI.y * qH0I.y
    + qBHr.s * qBI.z * qH0I.z
    - qBHr.x * qBI.s * qH0I.x
    + qBHr.x * qBI.x * qH0I.s
    + qBHr.x * qBI.y * qH0I.z
    - qBHr.x * qBI.z * qH0I.y
    - qBHr.y * qBI.s * qH0I.y
    - qBHr.y * qBI.x * qH0I.z
    + qBHr.y * qBI.y * qH0I.s
    + qBHr.y * qBI.z * qH0I.x
    - qBHr.z * qBI.s * qH0I.z
    + qBHr.z * qBI.x * qH0I.y
    - qBHr.z * qBI.y * qH0I.x
    + qBHr.z * qBI.z * qH0I.s
)

b = (
    -qBHr.s * qBI.s * qH0I.x
    + qBHr.s * qBI.x * qH0I.s
    + qBHr.s * qBI.y * qH0I.z
    - qBHr.s * qBI.z * qH0I.y
    - qBHr.x * qBI.s * qH0I.s
    - qBHr.x * qBI.x * qH0I.x
    - qBHr.x * qBI.y * qH0I.y
    - qBHr.x * qBI.z * qH0I.z
    - qBHr.y * qBI.s * qH0I.z
    + qBHr.y * qBI.x * qH0I.y
    - qBHr.y * qBI.y * qH0I.x
    + qBHr.y * qBI.z * qH0I.s
    + qBHr.z * qBI.s * qH0I.y
    + qBHr.z * qBI.x * qH0I.z
    - qBHr.z * qBI.y * qH0I.s
    - qBHr.z * qBI.z * qH0I.x
)

phi1 = 2 * (np.atan(b / a))
phi2 = 2 * (np.atan(b / a) + np.pi)
exp1 = a * np.cos(phi1 / 2) + b * np.sin(phi1 / 2)
exp2 = a * np.cos(phi2 / 2) + b * np.sin(phi2 / 2)

phi = phi1 if exp1 > exp2 else phi2

In [313]:
qHrH0 = Quaternion(phi=phi, axis=0)

# Calculate correct boresight quaternion
qBI = qBHr * qHrH0 * qH0I

In [314]:
# CODEX boresight frame quaternion in SPICE convention
q = qBI.spice

# Rotation matrix from J2000 to CODEX boresight frame
j2c = spice.q2m(q)

# Rotation matrix from CODEX boresight frame to J2000
c2j = j2c.transpose()

# Unit vector of the pointing direction in CODEX boresight frame (x axis)
u = np.array([1, 0, 0])

# Get RA and Dec of the pointing direction
cr, cra, cdec = spice.recrad(c2j.dot(u))

# Get pixel of Sun's center
X = np.degrees(radec2hpc(et, j2c, sunra, sundec))
P = crpix + rot(crota).dot(X[:, 0]) / cdelt

print("Sun coordinates: ", np.degrees(sunra), np.degrees(sundec))
print("CODEX B/I quaternion: ", q)
print("CODEX pointing: ", np.degrees(cra), np.degrees(cdec))
print("CODEX Sun's center: ", P)

Sun coordinates:  284.2738246385406 -22.787605752287693
CODEX B/I quaternion:  [ 0.01349001 -0.78324218  0.56595235  0.25699737]
CODEX pointing:  284.27308252708957 -22.787406905511325
CODEX Sun's center:  [1972.11398683 1075.13171364]


In [315]:
# Rotation matrix from IAU_SUN frame to j2000
s2j = spice.pxform("iau_sun", "j2000", et)

# Get the rotation matrix from IAU_SUN frame to CODEX boresight frame
s2c = j2c.dot(s2j)

# Unit vector of the Sun's rotation axis in the IAU_SUN frame (z axis)
a = np.array([0, 0, 1])

# Unit vector of the Sun's rotation axis in CODEX boresight frame
aprime = s2c.dot(a)

# Inclination of Sun's rotation axis w.r.t y axis in CODEX boresight frame
roll = np.arctan2(aprime[1], aprime[2])

print("CODEX roll (from vertical): ", 180 - np.degrees(roll + crota))

CODEX roll (from vertical):  20.659303467138727


In [316]:
header.append(("date-obs", header["obt_beg"]))
header.append(("wcsname", "Helioprojective-Cartesian"))
header.append(("ctype1", "hplt_tan"))
header.append(("ctype2", "hpln_tan"))
header.append(("cunit1", "arcsec"))
header.append(("cunit2", "arcsec"))
header.append(("crpix1", P[0]))
header.append(("crpix2", P[1]))
header.append(("crval1", 0))
header.append(("crval2", 0))
header.append(("cdelt1", cdelt * 3600))
header.append(("cdelt2", cdelt * 3600))
header.append(("pc1_1", np.cos(roll + crota)))
header.append(("pc1_2", np.sin(roll + crota)))
header.append(("pc2_1", -np.sin(roll + crota)))
header.append(("pc2_2", np.cos(roll + crota)))

hdu = fits.PrimaryHDU(data=image, header=header)
hdu_list = fits.HDUList([hdu])
hdu_list.writeto(input["l0_image"].replace("l0", "l1"), overwrite=True)